In [1]:
import pandas as pd 
import geopandas as gpd
import shapely.speedups
shapely.speedups.enable() 
import fiona 
import alphashape
from scipy.spatial import ConvexHull
import ogr 
import shapely
import matplotlib.pyplot as plt
import shapefile
from collections import OrderedDict
from pyproj import Transformer
from geopandas import GeoDataFrame
from shapely.geometry import Point
import pickle 

c:\users\ablanchi\appdata\local\programs\python\python36\lib\site-packages\geopandas\_compat.py:110: UserWarning: The Shapely GEOS version (3.8.0-CAPI-1.13.1 ) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [68]:
VILLE = "VILLEFRANCHE-SUR-MER"

In [69]:
#pd.read_csv("C:/Users/ablanchi/Desktop/lixels_cannes_test_more_50.csv", delimiter=';')
import pickle 

df = pd.read_excel("C:/GIT/2022/TOPONOMASCAPE/VILLE/"+VILLE+"/LIXELS_NETKDE_NEGATIF_0606.xlsx",engine='openpyxl')
## IMPORT NETWORK OR LIXELS
## IMPORT NETWORK OR LIXELS
streets = gpd.read_file("C:/GIT/2022/TOPONOMASCAPE/VILLE/"+VILLE+"/LIXELS_VILLE.shp")
streets = streets.to_crs("EPSG:2154")
lixels = streets[['lineID', 'uid', 'geometry']]
lixels
lixels1 = lixels.merge(df, on ="lineID")
lixels1
gdf1 = GeoDataFrame(lixels1, crs="EPSG:2154", geometry=lixels1.geometry_x)
gdf1

file = open("C:/GIT/2022/TOPONOMASCAPE/VILLE/"+VILLE+"/2_PHENOMENE_POSITIF.pickle", "rb")
gdf = pickle.load(file)
file.close()


gdf['lineID'] = gdf['lineID'].astype(int)

In [70]:
NETKDE_NEGATIF1 = gdf1
NETKDE_POSITIF1 = gdf

In [71]:
DROPTO = ['uid_x', 'troncon_de_x', 'geometry_x', 'uid_y',
       'troncon_de_y', 'troncon__1', 'troncon__2', 'troncon__3', 'NOM_count',
       'NOM_unique', 'NOM_empty', 'NOM_filled', 'NOM_min', 'NOM_max',
       'NOM_min_le', 'NOM_max_le', 'NOM_mean_l', 'geometry_y', 'geometry','uid_x', 'geometry_x', 'uid_y', 'troncon_de', 'troncon__1',
       'troncon__2', 'troncon__3', 'NOM_count', 'NOM_unique', 'NOM_empty',
       'NOM_filled', 'NOM_min', 'NOM_max', 'NOM_min_le', 'NOM_max_le',
       'NOM_mean_l', 'geometry_y', 'geometry', 'uid_x', 'geometry_x', 'uid_y', 'troncon_de', 'troncon__1',
       'troncon__2', 'troncon__3', 'NOM_count', 'NOM_unique', 'NOM_empty',
       'NOM_filled', 'NOM_min', 'NOM_max', 'NOM_min_le', 'NOM_max_le',
       'NOM_mean_l', 'geometry_y','uid_x', 'geometry_x', 'uid_y', 'troncon_de', 'troncon__1',
       'troncon__2', 'troncon__3', 'NOM_count', 'NOM_unique', 'NOM_empty',
       'NOM_filled', 'NOM_min', 'NOM_max', 'NOM_min_le', 'NOM_max_le',
       'NOM_mean_l', 'geometry_y']

In [72]:
# Dropping the columns
columns_to_drop = [col for col in DROPTO if col in NETKDE_POSITIF1.columns]
NETKDE_POSITIF1 = NETKDE_POSITIF1.drop(columns=columns_to_drop)
# Dropping the columns
columns_to_drop = [col for col in DROPTO if col in NETKDE_NEGATIF1.columns]
NETKDE_NEGATIF1 = NETKDE_NEGATIF1.drop(columns=columns_to_drop)


In [73]:
NETKDE_NEGATIF1 = NETKDE_NEGATIF1.set_index('lineID')
NETKDE_POSITIF1 = NETKDE_POSITIF1.set_index('lineID')

In [74]:
NETKDE_NEGATIF1= NETKDE_NEGATIF1.reindex(sorted(NETKDE_NEGATIF1.columns), axis=1)
NETKDE_POSITIF1 = NETKDE_POSITIF1.reindex(sorted(NETKDE_NEGATIF1.columns), axis=1)

In [75]:
df1 = pd.DataFrame(NETKDE_POSITIF1)
df2 = pd.DataFrame(NETKDE_NEGATIF1)

In [76]:
assert len(set(df1.index)-set(df2.index))==0
assert len(set(df2.index)-set(df1.index))==0
assert len(set(df1.columns)-set(df2.columns))==0
assert len(set(df2.columns)-set(df1.columns))==0


df1 = pd.melt(df1, ignore_index=False).reset_index().set_index(['lineID', 'variable'])
df2 = pd.melt(df2, ignore_index=False).reset_index().set_index(['lineID', 'variable'])
df3 = df1.join(df2, lsuffix='_p', rsuffix='_n')

In [77]:
#Ratio_toponyme = Kernel_positif_toponyme / (Kernel_positif_toponyme + Kernel_negatif_toponyme)
df3['ratio'] = df3['value_p'] / (df3['value_p'] + df3['value_n'])
df3 = df3[['ratio']].reset_index().pivot(index='lineID', columns='variable')['ratio']

In [78]:
NETKDE = gpd.GeoDataFrame(lixels, geometry="geometry")
NETKDE = NETKDE[['lineID','uid','geometry' ]]
NETKDE_RATIO = NETKDE.merge(df3, on='lineID')

In [79]:
notcol = ('lineID','uid','geometry' )

In [80]:
for col_name in NETKDE_RATIO.columns:
    # Exclure la colonne 
    if col_name not in notcol:
        print(col_name)
        list_idx_positif = list(gdf.loc[gdf[col_name] > 0.0].lineID)
        #list_idx_positif = list(gdf.loc[gdf[col_name]].int(lineID))
        
## DELATE LIXELS NOT IN INTERVALE 
        NETKDE_RATIO[col_name] = NETKDE_RATIO.apply(lambda x : x[col_name] if (x['lineID'] in list_idx_positif) else 0, axis=1)

BONAPART
beaulieu mer
cap ferrat
corne or
jean cap ferrat
monaco
villefranche
vinaigrier


In [81]:
with open("C:/GIT/2022/TOPONOMASCAPE/VILLE/"+VILLE+"/Ratio.pickle", 'wb') as f:
    pickle.dump(NETKDE_RATIO, f)


In [82]:
NETKDE_RATIO.to_excel("C:/GIT/2022/TOPONOMASCAPE/VILLE/"+VILLE+"/NETKDE_RATIO.xlsx")